In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

# url=os.getenv("NEO4J_URI")
url="neo4j+ssc://bcb8919a.databases.neo4j.io"
username=os.getenv("NEO4J_USERNAME")
password=os.getenv("NEO4J_PASSWORD")
database=os.getenv("NEO4J_DATABASE")

In [3]:
# from langchain_neo4j import Neo4jGraph
from langchain_community.graphs import Neo4jGraph
graph = Neo4jGraph(url=url,username=username,password=password,database=database)
graph

C:\Users\HP\AppData\Local\Temp\ipykernel_12976\1995070383.py:3: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(url=url,username=username,password=password,database=database)


In [4]:
from neo4j import GraphDatabase

url = "neo4j+ssc://bcb8919a.databases.neo4j.io"

driver = GraphDatabase.driver(
    url,
    auth=(username, password)
)

with driver.session() as session:
    print(session.run("RETURN 1").single())

<Record 1=1>


In [5]:
graph.query("RETURN 1 AS num")

[{'num': 1}]

In [11]:
movie_query ="""
LOAD CSV WITH HEADERS FROM 
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' 
as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

In [12]:
graph.query(movie_query)

[]

In [13]:
graph.query("MATCH (n) RETURN count(n)")

[{'count(n)': 1564}]

In [14]:
graph.query("""
MATCH (m:Movie)
RETURN m.title, m.imdbRating
LIMIT 5
""")

[{'m.title': 'Toy Story', 'm.imdbRating': 8.3},
 {'m.title': 'Jumanji', 'm.imdbRating': 6.9},
 {'m.title': 'Grumpier Old Men', 'm.imdbRating': 6.6},
 {'m.title': 'Waiting to Exhale', 'm.imdbRating': 5.6},
 {'m.title': 'Father of the Bride Part II', 'm.imdbRating': 5.9}]

In [15]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # fast + good enough
    groq_api_key=os.getenv("GROQ_API_KEY")
)

In [18]:
from langchain_classic.chains import GraphCypherQAChain
chain = GraphCypherQAChain.from_llm(llm=llm,graph=graph,verbose=True,allow_dangerous_requests=True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x000001B15412ACC0>, cypher_generation_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}'), llm=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_

In [30]:
chain.invoke("Who directed Inception?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: 'Inception'})-[:DIRECTED]->(p:Person) RETURN p
Full Context:
[]

> Finished chain.


{'query': 'Who directed Inception?', 'result': "I don't know the answer."}

In [22]:
chain.invoke({"query":"Who was the director of the movie Casino"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:WORKS_IN]->(w:Working)<-[:WORKS_IN]-(d:Person) WHERE w.country = 'USA' AND d.name = 'Martin Scorsese' RETURN p.name
Full Context:
[]

> Finished chain.


{'query': 'Who was the director of the movie Casino',
 'result': "I don't know the answer."}

In [21]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Person {DOB: INTEGER, POB: STRING, name: STRING, dob: INTEGER}
Working {country: STRING}
Country {name: STRING, code: STRING}
Skill {name: STRING, type: STRING}
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Genre {name: STRING}
Relationship properties:
WORKS_IN {since: INTEGER}
HAS_SKILL {level: STRING}
The relationships:
(:Person)-[:WORKS_IN]->(:Working)
(:Person)-[:LIVES_IN]->(:Country)
(:Person)-[:HAS_SKILL]->(:Skill)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


In [31]:
from langchain_core.prompts import PromptTemplate

CYPHER_PROMPT = PromptTemplate.from_template("""
You are an expert Neo4j Cypher generator.

Use ONLY this schema:

(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)

Rules:
- ALWAYS follow relationship direction exactly as given
- DIRECTED goes from Person → Movie
- ACTED_IN goes from Person → Movie
- NEVER reverse relationship direction
- Use CONTAINS for movie title matching (not exact match)

Question:
{question}

Return ONLY Cypher query.
""")

In [32]:
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=CYPHER_PROMPT,
    verbose=True,
    allow_dangerous_requests=True
)

In [33]:
chain.invoke({"query":"Who was the director of the movie Casino"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie)
WHERE m.title CONTAINS 'Casino'
RETURN p
Full Context:
[{'p': {'name': 'Martin Scorsese'}}]

> Finished chain.


{'query': 'Who was the director of the movie Casino',
 'result': 'Martin Scorsese was the director of the movie Casino.'}

In [34]:
chain.invoke("Who directed Inception?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (d:Person)-[:DIRECTED]->(m:Movie) WHERE m.title CONTAINS "Inception" RETURN d
Full Context:
[]

> Finished chain.


{'query': 'Who directed Inception?', 'result': "I don't know the answer."}

In [35]:
graph.query("""
MATCH (m:Movie)
WHERE m.title CONTAINS "Inception"
RETURN m.title
""")

[]

In [36]:
chain.invoke("Who directed Toy story?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie) WHERE m.title CONTAINS 'Toy Story' RETURN p
Full Context:
[{'p': {'name': 'John Lasseter'}}]

> Finished chain.


{'query': 'Who directed Toy story?',
 'result': 'John Lasseter directed Toy story.'}

In [37]:
print(graph.schema)

Node properties:
Person {DOB: INTEGER, POB: STRING, name: STRING, dob: INTEGER}
Working {country: STRING}
Country {name: STRING, code: STRING}
Skill {name: STRING, type: STRING}
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Genre {name: STRING}
Relationship properties:
WORKS_IN {since: INTEGER}
HAS_SKILL {level: STRING}
The relationships:
(:Person)-[:WORKS_IN]->(:Working)
(:Person)-[:LIVES_IN]->(:Country)
(:Person)-[:HAS_SKILL]->(:Skill)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
